# Notebook 18: Deep Feedforward Networks – Backprop, Optimizers, Callbacks
### Part 18/30 – ML Mastery Series for Python Experts

You built your first nets — now let's make them deeper, faster, and more stable. Time to master the dark art of training deep feedforward models…

## Backpropagation – How Networks Actually Learn

Understanding backpropagation is crucial for debugging deep networks. Here's what happens under the hood:

- **Chain Rule Application**: Gradients flow backward through the network using the chain rule from calculus — $\frac{\partial L}{\partial w} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} \cdot \frac{\partial z}{\partial w}$
- **Forward vs Backward Pass**: Forward pass computes predictions and stores intermediate activations; backward pass propagates error gradients back to update weights
- **Gradient Flow**: In deep networks, gradients can shrink (vanish) or explode exponentially as they propagate through many layers
- **Vanishing Gradients**: Common with sigmoid/tanh activations where derivatives saturate near 0, making early layers learn extremely slowly
- **Exploding Gradients**: Large weight initializations or high learning rates cause gradients to grow exponentially, leading to unstable updates and NaN losses
- **Weight Updates**: Optimizers use gradients to adjust weights: $w_{new} = w_{old} - \eta \cdot \nabla_w L$ (basic SGD), with more sophisticated rules for adaptive methods
- **Why Deeper is Harder**: Each additional layer compounds gradient flow issues; without proper initialization or normalization, deep nets become untrainable
- **Modern Solutions**: ReLU activations, BatchNorm, residual connections, and careful initialization (He, Xavier) help maintain healthy gradient flow

## Learning Objectives

By the end of this notebook, you will be able to:

- **Understand backprop conceptually** — trace how gradients flow from loss to weights through the computational graph
- **Compare optimizers and their convergence behavior** — know when to use SGD+momentum vs Adam vs AdamW
- **Apply L1/L2 regularization and weight decay** — control model complexity and prevent overfitting
- **Use dropout and batch/layer normalization** — stabilize training and enable deeper architectures
- **Build and train models >10 layers deep** — stack Dense layers with proper regularization
- **Leverage advanced callbacks** — TensorBoard visualization, learning rate scheduling, and model checkpointing
- **Monitor gradient norms and activations** — diagnose training issues before they become critical
- **Identify and fix training pathologies** — loss spikes, plateaus, dead neurons, and overfitting patterns
- **Implement gradient clipping strategies** — prevent exploding gradients in recurrent or very deep networks
- **Design complete training pipelines** — combine all techniques for production-ready model training

## ⚡ 1. Deeper Network on MNIST (Digits) – Baseline

Let's establish a baseline with a deeper fully-connected network on the classic MNIST digits dataset. This will be our reference point for comparing optimizers and regularization techniques.

In [ ]:
# Standard imports for deep learning workflow
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from tensorflow.keras.datasets import mnist, fashion_mnist, cifar10
from tensorflow.keras.optimizers import SGD, RMSprop, Adam, Nadam
from tensorflow.keras.optimizers.experimental import AdamW

# Set visualization style and random seeds for reproducibility
%matplotlib inline
sns.set_theme(style="whitegrid", palette="husl")
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load MNIST dataset - 60k training, 10k test images of handwritten digits 0-9
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize pixel values to [0, 1] range for stable gradient flow
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Flatten 28x28 images to 784-dimensional vectors for dense layers
x_train_flat = x_train.reshape(-1, 28 * 28)
x_test_flat = x_test.reshape(-1, 28 * 28)

print(f"Training set shape: {x_train_flat.shape}, Labels: {y_train.shape}")
print(f"Test set shape: {x_test_flat.shape}, Labels: {y_test.shape}")
print(f"Pixel value range: [{x_train_flat.min():.2f}, {x_train_flat.max():.2f}]")

In [ ]:
# Build deeper baseline network with 6 hidden layers
# Architecture: 784 → 256 → 128 → 128 → 64 → 64 → 32 → 10
baseline_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu', name='hidden_1'),
    layers.Dense(128, activation='relu', name='hidden_2'),
    layers.Dense(128, activation='relu', name='hidden_3'),
    layers.Dense(64, activation='relu', name='hidden_4'),
    layers.Dense(64, activation='relu', name='hidden_5'),
    layers.Dense(32, activation='relu', name='hidden_6'),
    layers.Dense(10, activation='softmax', name='output')  # 10 digit classes
], name='baseline_deep_net')

# Compile with Adam (adaptive learning rates) and sparse labels (integer indices)
baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display architecture summary
baseline_model.summary()

In [ ]:
# Train the baseline model for 30 epochs with 20% validation split
history_baseline = baseline_model.fit(
    x_train_flat, y_train,
    epochs=30,
    validation_split=0.2,  # 12k samples for validation
    batch_size=128,        # Mini-batch size for gradient estimates
    verbose=1
)

print(f"\nFinal training accuracy: {history_baseline.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history_baseline.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Plot training curves to visualize convergence and overfitting patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history_baseline.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_baseline.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Loss Curves - Baseline Deep Network', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Sparse Categorical Crossentropy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history_baseline.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history_baseline.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Accuracy Curves - Baseline Deep Network', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluate on test set
test_loss, test_acc = baseline_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

## ⚡ 2. Optimizers Comparison – Convergence Speed & Stability

Different optimizers exhibit distinct convergence characteristics. We'll compare:
- **SGD**: Classic stochastic gradient descent
- **SGD + Momentum (0.9)**: Accelerates through consistent directions, dampens oscillations
- **RMSprop**: Adapts learning rates per-parameter based on recent gradient magnitudes
- **Adam**: Combines momentum and RMSprop ideas (adaptive moments)
- **Nadam**: Adam with Nesterov momentum for better lookahead
- **AdamW**: Adam with proper weight decay decoupled from gradient updates

Key insight: Adam converges faster early but may generalize worse than well-tuned SGD+momentum.

In [ ]:
# Dictionary to store results for comparison
optimizer_results = {}

# Define optimizers to compare with their specific parameters
optimizers_dict = {
    'SGD': SGD(learning_rate=0.01),
    'SGD+Momentum': SGD(learning_rate=0.01, momentum=0.9),
    'RMSprop': RMSprop(learning_rate=0.001),
    'Adam': Adam(learning_rate=0.001),
    'Nadam': Nadam(learning_rate=0.001),
    'AdamW': AdamW(learning_rate=0.001, weight_decay=0.004)
}

# Train identical architecture with each optimizer
for opt_name, optimizer in optimizers_dict.items():
    print(f"\n{'='*50}")
    print(f"Training with {opt_name}...")
    print(f"{'='*50}")
    
    # Rebuild identical model architecture for fair comparison
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    # Train for 20 epochs (sufficient to see convergence differences)
    history = model.fit(
        x_train_flat, y_train,
        epochs=20,
        validation_split=0.2,
        batch_size=128,
        verbose=0  # Suppress output for cleaner logs
    )
    
    optimizer_results[opt_name] = history.history
    print(f"Final val accuracy: {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Visualize optimizer comparison - validation loss and accuracy
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = sns.color_palette("husl", len(optimizer_results))

# Plot validation loss for all optimizers
for idx, (name, hist) in enumerate(optimizer_results.items()):
    axes[0].plot(hist['val_loss'], label=name, linewidth=2.5, color=colors[idx])
axes[0].set_title('Validation Loss Comparison by Optimizer', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')  # Log scale to see early differences

# Plot validation accuracy
for idx, (name, hist) in enumerate(optimizer_results.items()):
    axes[1].plot(hist['val_accuracy'], label=name, linewidth=2.5, color=colors[idx])
axes[1].set_title('Validation Accuracy Comparison by Optimizer', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].legend(loc='lower right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\nOptimizer Performance Summary (Final Validation Metrics):")
print("-" * 60)
for name, hist in optimizer_results.items():
    final_val_acc = hist['val_accuracy'][-1]
    final_val_loss = hist['val_loss'][-1]
    print(f"{name:15s} | Val Acc: {final_val_acc:.4f} | Val Loss: {final_val_loss:.4f}")

## ⚡ 3. Regularization Techniques – L1/L2 & Dropout

Deep networks easily overfit training data. We apply:
- **L2 Regularization (Ridge)**: Adds $\lambda \sum w^2$ to loss, penalizes large weights
- **L1 Regularization (Lasso)**: Adds $\lambda \sum |w|$, encourages sparsity
- **Dropout**: Randomly zeroes neurons during training (rate 0.3-0.5), forces redundant representations

Compare: No regularization → quick overfitting vs. regularized → slower training but better generalization.

In [ ]:
# Build overfitting-prone model (no regularization) for comparison
overfit_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, activation='relu'),  # Larger capacity to encourage overfitting
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

overfit_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train without regularization
history_overfit = overfit_model.fit(
    x_train_flat, y_train,
    epochs=50,  # Train longer to see overfitting
    validation_split=0.2,
    batch_size=128,
    verbose=0
)

print(f"Overfit model - Final train acc: {history_overfit.history['accuracy'][-1]:.4f}")
print(f"Overfit model - Final val acc: {history_overfit.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Build regularized model with L2 and Dropout
regularized_model = keras.Sequential([
    layers.Input(shape=(784,)),
    # L2 regularization: kernel_regularizer adds penalty to loss for large weights
    layers.Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),  # Randomly drop 30% of neurons during training
    
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    
    layers.Dense(10, activation='softmax')
])

regularized_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train with regularization
history_reg = regularized_model.fit(
    x_train_flat, y_train,
    epochs=50,
    validation_split=0.2,
    batch_size=128,
    verbose=0
)

print(f"Regularized model - Final train acc: {history_reg.history['accuracy'][-1]:.4f}")
print(f"Regularized model - Final val acc: {history_reg.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Compare overfitting vs regularized training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loss comparison
axes[0].plot(history_overfit.history['loss'], label='Overfit - Train', linestyle='--', alpha=0.7)
axes[0].plot(history_overfit.history['val_loss'], label='Overfit - Val', linewidth=2, color='red')
axes[0].plot(history_reg.history['loss'], label='Regularized - Train', linestyle='--', alpha=0.7)
axes[0].plot(history_reg.history['val_loss'], label='Regularized - Val', linewidth=2, color='green')
axes[0].set_title('Loss: Overfitting vs Regularization', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy comparison
axes[1].plot(history_overfit.history['accuracy'], label='Overfit - Train', linestyle='--', alpha=0.7)
axes[1].plot(history_overfit.history['val_accuracy'], label='Overfit - Val', linewidth=2, color='red')
axes[1].plot(history_reg.history['accuracy'], label='Regularized - Train', linestyle='--', alpha=0.7)
axes[1].plot(history_reg.history['val_accuracy'], label='Regularized - Val', linewidth=2, color='green')
axes[1].set_title('Accuracy: Overfitting vs Regularization', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate generalization gap (train - val accuracy)
overfit_gap = history_overfit.history['accuracy'][-1] - history_overfit.history['val_accuracy'][-1]
reg_gap = history_reg.history['accuracy'][-1] - history_reg.history['val_accuracy'][-1]
print(f"\nGeneralization Gap (Train - Val Accuracy):")
print(f"Overfit model: {overfit_gap:.4f} (higher = more overfitting)")
print(f"Regularized model: {reg_gap:.4f}")

In [ ]:
# Visualize weight distributions to see effect of L2 regularization
def get_weights_histogram(model):
    """Extract all dense layer weights into a flat array"""
    weights = []
    for layer in model.layers:
        if isinstance(layer, layers.Dense):
            w = layer.get_weights()[0]  # Get kernel weights (not biases)
            weights.extend(w.flatten())
    return np.array(weights)

overfit_weights = get_weights_histogram(overfit_model)
reg_weights = get_weights_histogram(regularized_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Weight histograms
axes[0].hist(overfit_weights, bins=100, alpha=0.7, label='Overfit Model', density=True, color='red')
axes[0].hist(reg_weights, bins=100, alpha=0.7, label='L2 Regularized', density=True, color='green')
axes[0].set_title('Weight Distribution Comparison', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Weight Value')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Weight magnitude statistics
axes[1].bar(['Overfit', 'L2 Regularized'], 
            [np.mean(np.abs(overfit_weights)), np.mean(np.abs(reg_weights))],
            color=['red', 'green'], alpha=0.7)
axes[1].set_title('Mean Absolute Weight Value', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Mean |Weight|')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Weight statistics:")
print(f"Overfit model - Mean abs weight: {np.mean(np.abs(overfit_weights)):.4f}, Std: {np.std(overfit_weights):.4f}")
print(f"Regularized model - Mean abs weight: {np.mean(np.abs(reg_weights)):.4f}, Std: {np.std(reg_weights):.4f}")

## ⚡ 4. Normalization Layers – BatchNorm & LayerNorm

Normalization layers stabilize training by maintaining consistent input distributions:

**Batch Normalization**: 
$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$, then $\gamma \hat{x} + \beta$
- Normalizes across the batch dimension
- Allows higher learning rates, reduces sensitivity to initialization
- Acts as regularization (noise from batch statistics)

**Layer Normalization**:
- Normalizes across feature dimension (per sample)
- Better for small batches or RNNs/Transformers
- Used in modern architectures like BERT, GPT

In [ ]:
# Build model WITHOUT BatchNorm for comparison
no_bn_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

no_bn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_no_bn = no_bn_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=128,
    verbose=0
)

print("No BatchNorm - Training complete")

In [ ]:
# Build model WITH BatchNormalization
# Note: BatchNorm goes BEFORE activation (modern best practice)
bn_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, use_bias=False),  # Bias not needed with BatchNorm
    layers.BatchNormalization(),        # Normalize activations
    layers.Activation('relu'),          # Then apply nonlinearity
    
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(64, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(10, activation='softmax')
])

bn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_bn = bn_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=128,
    verbose=0
)

print("With BatchNorm - Training complete")

In [ ]:
# Compare convergence speed with and without BatchNorm
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Validation loss
axes[0].plot(history_no_bn.history['val_loss'], label='Without BatchNorm', linewidth=2.5)
axes[0].plot(history_bn.history['val_loss'], label='With BatchNorm', linewidth=2.5)
axes[0].set_title('Validation Loss: BatchNorm Effect', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Validation accuracy
axes[1].plot(history_no_bn.history['val_accuracy'], label='Without BatchNorm', linewidth=2.5)
axes[1].plot(history_bn.history['val_accuracy'], label='With BatchNorm', linewidth=2.5)
axes[1].set_title('Validation Accuracy: BatchNorm Effect', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Show convergence speed comparison
print("\nEpochs to reach 90% validation accuracy:")
try:
    no_bn_epoch = next(i for i, acc in enumerate(history_no_bn.history['val_accuracy']) if acc >= 0.90)
    print(f"Without BatchNorm: {no_bn_epoch + 1} epochs")
except StopIteration:
    print(f"Without BatchNorm: Did not reach 90% (max: {max(history_no_bn.history['val_accuracy']):.4f})")

try:
    bn_epoch = next(i for i, acc in enumerate(history_bn.history['val_accuracy']) if acc >= 0.90)
    print(f"With BatchNorm: {bn_epoch + 1} epochs")
except StopIteration:
    print(f"With BatchNorm: Did not reach 90% (max: {max(history_bn.history['val_accuracy']):.4f})")

In [ ]:
# Demonstrate LayerNormalization (alternative to BatchNorm)
# Particularly useful for small batches or sequence models
layernorm_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256),
    layers.LayerNormalization(),  # Normalize across features instead of batch
    layers.Activation('relu'),
    
    layers.Dense(128),
    layers.LayerNormalization(),
    layers.Activation('relu'),
    
    layers.Dense(10, activation='softmax')
])

layernorm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history_ln = layernorm_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.2,
    batch_size=128,
    verbose=0
)

print(f"LayerNorm model - Final val accuracy: {history_ln.history['val_accuracy'][-1]:.4f}")
print("\nNote: LayerNorm is particularly effective in:")
print("- Recurrent neural networks (LSTM/GRU)")
print("- Transformer architectures")
print("- Situations with small batch sizes where BatchNorm statistics are noisy")


## ⚡ 5. Advanced Callbacks & Monitoring

Callbacks are powerful tools for controlling training dynamically:
- **TensorBoard**: Real-time visualization of metrics, histograms, and computational graphs
- **EarlyStopping**: Halt training when validation metric stops improving (prevents overfitting)
- **ReduceLROnPlateau**: Automatically decay learning rate when progress stalls
- **LearningRateScheduler**: Custom learning rate annealing (warmup, exponential decay, etc.)
- **ModelCheckpoint**: Save best model weights during training
- **CSVLogger**: Persist training history to disk for analysis

In [ ]:
import os
import datetime

# Create directories for logs and checkpoints
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# Define comprehensive callback setup
callback_list = [
    # TensorBoard visualization - launch with: tensorboard --logdir logs/fit
    callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=1,  # Log weight histograms every epoch
        update_freq='epoch',
        profile_batch=0    # Disable profiling for speed
    ),
    
    # Stop training if val_loss doesn't improve for 10 epochs
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,  # Roll back to best epoch
        verbose=1
    ),
    
    # Reduce learning rate when plateau detected
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,      # Multiply LR by 0.5
        patience=5,      # Wait 5 epochs before reducing
        min_lr=1e-7,     # Don't go below this
        verbose=1
    ),
    
    # Custom learning rate schedule: warmup then exponential decay
    callbacks.LearningRateScheduler(
        lambda epoch, lr: lr * 0.95 if epoch > 5 else lr * 1.05,  # Warmup then decay
        verbose=0
    ),
    
    # Save best model based on validation accuracy
    callbacks.ModelCheckpoint(
        filepath=os.path.join(checkpoint_dir, 'best_model.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        save_weights_only=False,
        verbose=1
    ),
    
    # Log to CSV for external analysis
    callbacks.CSVLogger('training_log.csv', append=False)
]

print(f"Callbacks configured:")
print(f"- TensorBoard logs: {log_dir}")
print(f"- Checkpoints: {checkpoint_dir}/best_model.keras")
print(f"- CSV log: training_log.csv")

In [ ]:
# Build a deeper model to demonstrate callbacks effectively
callback_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(512, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    layers.Dense(256, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    layers.Dense(128, use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
], name='callback_demo_model')

callback_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model ready for training with callbacks...")

In [ ]:
# Train with full callback suite
history_callbacks = callback_model.fit(
    x_train_flat, y_train,
    epochs=50,  # Will likely stop early due to EarlyStopping
    validation_split=0.2,
    batch_size=128,
    callbacks=callback_list,
    verbose=1
)

print(f"\nTraining stopped at epoch {len(history_callbacks.history['loss'])}")
print(f"Best validation accuracy: {max(history_callbacks.history['val_accuracy']):.4f}")

In [ ]:
# Visualize callback effects: learning rate changes and early stopping
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves with early stopping point marked
axes[0].plot(history_callbacks.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history_callbacks.history['val_loss'], label='Validation Loss', linewidth=2)
best_epoch = np.argmin(history_callbacks.history['val_loss'])
axes[0].axvline(best_epoch, color='red', linestyle='--', label=f'Best Epoch ({best_epoch})')
axes[0].set_title('Loss with Early Stopping', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Learning rate schedule
axes[1].plot(history_callbacks.history.get('lr', []), linewidth=2, color='purple')
axes[1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

# Accuracy curves
axes[2].plot(history_callbacks.history['accuracy'], label='Training', linewidth=2)
axes[2].plot(history_callbacks.history['val_accuracy'], label='Validation', linewidth=2)
axes[2].axvline(best_epoch, color='red', linestyle='--', alpha=0.7)
axes[2].set_title('Accuracy with Best Model Checkpoint', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Load and evaluate the best saved model
best_model = keras.models.load_model('checkpoints/best_model.keras')
test_loss, test_acc = best_model.evaluate(x_test_flat, y_test, verbose=0)
print(f"\nBest saved model test accuracy: {test_acc:.4f}")

## ⚡ 6. Gradient Clipping & Monitoring Gradient Flow

In deep networks, gradients can explode (grow very large) causing unstable training. Solutions:
- **Gradient Clipping**: Limit gradient norms or values during backprop
  - `clipnorm=1.0`: Rescale if L2 norm exceeds threshold
  - `clipvalue=0.5`: Clip individual values to range [-0.5, 0.5]
- **Gradient Monitoring**: Track gradient norms per layer to diagnose vanishing/exploding gradients

In [ ]:
# Custom callback to log gradient norms per layer during training
class GradientNormLogger(callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.gradient_norms = {f'layer_{i}': [] for i in range(10)}  # Reserve space
        self.epoch_gradients = []
    
    def on_epoch_end(self, epoch, logs=None):
        # Get a sample batch to compute gradients
        sample_size = min(1024, len(x_train_flat))
        x_sample = x_train_flat[:sample_size]
        y_sample = y_train[:sample_size]
        
        with tf.GradientTape() as tape:
            predictions = self.model(x_sample, training=True)
            loss = self.model.compiled_loss(y_sample, predictions)
        
        # Compute gradients
        gradients = tape.gradient(loss, self.model.trainable_variables)
        
        # Log norms for each layer's weights
        layer_idx = 0
        for var, grad in zip(self.model.trainable_variables, gradients):
            if grad is not None and 'kernel' in var.name:
                norm = tf.norm(grad).numpy()
                layer_name = f'layer_{layer_idx}'
                if layer_name not in self.gradient_norms:
                    self.gradient_norms[layer_name] = []
                self.gradient_norms[layer_name].append(norm)
                layer_idx += 1
        
        # Also log average gradient norm
        valid_grads = [g for g in gradients if g is not None]
        if valid_grads:
            avg_norm = np.mean([tf.norm(g).numpy() for g in valid_grads])
            self.epoch_gradients.append(avg_norm)

# Instantiate the logger
grad_logger = GradientNormLogger()

In [ ]:
# Build model with gradient clipping to prevent explosion
# clipnorm=1.0 ensures total gradient L2 norm doesn't exceed 1.0
clipped_optimizer = Adam(learning_rate=0.001, clipnorm=1.0)

grad_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

grad_model.compile(
    optimizer=clipped_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train with gradient monitoring
history_grad = grad_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=256,  # Larger batch for stable gradient estimates
    callbacks=[grad_logger],
    verbose=0
)

print(f"Training complete - monitored {len(grad_logger.epoch_gradients)} epochs")

In [ ]:
# Visualize gradient flow across layers and epochs
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot average gradient norm over epochs
axes[0].plot(grad_logger.epoch_gradients, linewidth=2, color='blue')
axes[0].set_title('Average Gradient Norm Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Average L2 Gradient Norm')
axes[0].axhline(1.0, color='red', linestyle='--', label='Clip Threshold (clipnorm=1.0)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot gradient norms by layer (heatmap)
# Prepare data: layers x epochs
layer_names = sorted([k for k in grad_logger.gradient_norms.keys() if grad_logger.gradient_norms[k]])
if layer_names:
    grad_matrix = np.array([grad_logger.gradient_norms[name] for name in layer_names])
    
    im = axes[1].imshow(grad_matrix, aspect='auto', cmap='viridis', interpolation='nearest')
    axes[1].set_title('Gradient Norms by Layer Over Time', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Layer Index')
    axes[1].set_yticks(range(len(layer_names)))
    axes[1].set_yticklabels(layer_names)
    plt.colorbar(im, ax=axes[1], label='Gradient L2 Norm')

plt.tight_layout()
plt.show()

print(f"\nGradient Statistics:")
print(f"Mean gradient norm: {np.mean(grad_logger.epoch_gradients):.4f}")
print(f"Max gradient norm: {np.max(grad_logger.epoch_gradients):.4f}")
print(f"Final gradient norm: {grad_logger.epoch_gradients[-1]:.4f}")

In [ ]:
# Compare training stability: Clipped vs Unclipped gradients
# Build identical model without clipping
unclipped_model = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])

unclipped_model.compile(
    optimizer=Adam(learning_rate=0.001),  # No clipping
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_unclipped = unclipped_model.fit(
    x_train_flat, y_train,
    epochs=20,
    validation_split=0.2,
    batch_size=256,
    verbose=0
)

# Compare loss curves
plt.figure(figsize=(12, 5))
plt.plot(history_unclipped.history['loss'], label='No Clipping', linewidth=2)
plt.plot(history_grad.history['loss'], label='Clipnorm=1.0', linewidth=2)
plt.title('Training Stability: Gradient Clipping Effect', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final loss without clipping: {history_unclipped.history['loss'][-1]:.4f}")
print(f"Final loss with clipping: {history_grad.history['loss'][-1]:.4f}")

## ⚡ 7. Putting It Together – Deep Net on Fashion-MNIST

Now we combine all techniques on a harder dataset: Fashion-MNIST (10 clothing categories).
We'll build a deep network (10+ layers) using:
- BatchNorm for stable training
- Dropout for regularization
- L2 weight decay
- AdamW optimizer
- Advanced callbacks for monitoring

In [ ]:
# Load Fashion-MNIST (more challenging than digits)
(fashion_train, fashion_labels_train), (fashion_test, fashion_labels_test) = fashion_mnist.load_data()

# Preprocess
fashion_train = fashion_train.astype('float32') / 255.0
fashion_test = fashion_test.astype('float32') / 255.0
fashion_train_flat = fashion_train.reshape(-1, 28 * 28)
fashion_test_flat = fashion_test.reshape(-1, 28 * 28)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Fashion-MNIST shape: {fashion_train_flat.shape}")
print(f"Classes: {class_names}")

In [ ]:
# Build deep network with all modern techniques
deep_fashion_model = keras.Sequential([
    layers.Input(shape=(784,)),
    
    # Block 1
    layers.Dense(512, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Block 2
    layers.Dense(256, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Block 3
    layers.Dense(128, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Block 4
    layers.Dense(128, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Block 5
    layers.Dense(64, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.3),
    
    # Block 6
    layers.Dense(64, use_bias=False, kernel_regularizer=regularizers.l2(0.0001)),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    
    # Block 7
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.0001)),
    
    # Output
    layers.Dense(10, activation='softmax')
], name='deep_fashion_net')

# Use AdamW with weight decay and gradient clipping
adamw_optimizer = AdamW(
    learning_rate=0.001,
    weight_decay=0.001,
    clipnorm=1.0
)

deep_fashion_model.compile(
    optimizer=adamw_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

deep_fashion_model.summary()

In [ ]:
# Setup callbacks for this training run
fashion_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'fashion_best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train the deep model
history_fashion = deep_fashion_model.fit(
    fashion_train_flat, fashion_labels_train,
    epochs=100,  # Will early stop
    validation_split=0.2,
    batch_size=256,
    callbacks=fashion_callbacks,
    verbose=1
)

print(f"\nTraining completed at epoch {len(history_fashion.history['loss'])}")

In [ ]:
# Evaluate and visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Training curves
axes[0].plot(history_fashion.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history_fashion.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('Fashion-MNIST: Deep Network Performance', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curves
axes[1].plot(history_fashion.history['loss'], label='Train', linewidth=2)
axes[1].plot(history_fashion.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Fashion-MNIST: Loss Curves', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Final evaluation
test_loss, test_acc = deep_fashion_model.evaluate(fashion_test_flat, fashion_labels_test, verbose=0)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")
print(f"Final Test Loss: {test_loss:.4f}")

# Compare to baseline (shallow model performance would be ~0.85-0.87)
print(f"\nImprovement over typical shallow baseline: ~{(test_acc - 0.87)*100:.1f} percentage points")

## Common Pitfalls & Pro Tips

**Training Dynamics:**
- ⚠️ **Adam converges too fast** → Poor generalization. Try SGD+momentum with cosine decay for final epochs
- ⚠️ **No BatchNorm in deep nets** → Unstable training, sensitive to initialization. Always use normalization for >3 hidden layers
- ⚠️ **Dropout at test time** → Catastrophic performance drop. Keras handles this automatically, but be careful with custom inference loops

**Learning Rate Management:**
- ⚠️ **Huge LR after warmup** → Loss explosion. Use learning rate scheduling with plateau detection
- ⚠️ **Fixed LR throughout training** → Suboptimal convergence. Always use decay or adaptive scheduling
- ⚠️ **LR too small with BatchNorm** → BatchNorm statistics become noisy. Use LR at least 1e-4 with default epsilon

**Regularization Traps:**
- ⚠️ **L2 too strong** → Underfitting. Monitor weight magnitudes; if all weights < 0.01, reduce regularization
- ⚠️ **Dropout + L2 together** → Can be redundant. Start with one, add other if still overfitting
- ⚠️ **High dropout on small datasets** → Removes too much signal. Use <0.3 for n < 10k samples

**Architecture & Debugging:**
- ⚠️ **Not monitoring gradients** → Silent vanishing/exploding. Log gradient norms every epoch
- ⚠️ **Dead ReLU neurons** → Large negative biases. Use LeakyReLU or PReLU for very deep nets
- ⚠️ **BatchNorm before activation (old way)** → Works but suboptimal. Use Dense → BN → Activation order
- ⚠️ **Inconsistent batch sizes** → BatchNorm statistics vary wildly. Fix batch size or use LayerNorm instead

**Data & Preprocessing:**
- ⚠️ **Unnormalized inputs** → Gradients explode, training fails. Always scale to [0,1] or standardize
- ⚠️ **Class imbalance without handling** → Biased predictions. Use class weights or focal loss
- ⚠️ **No validation set monitoring** → Overfitting undetected. Always use validation_split or validation_data

## Exercises

**Easy:**
1. Add BatchNorm + Dropout to the breast cancer model from Notebook 17. Compare training curves with and without these layers. Does validation accuracy improve?

**Medium:**
2. Train Fashion-MNIST with Adam (lr=0.001) vs SGD+momentum (lr=0.01, momentum=0.9) for 30 epochs each. Plot validation accuracy curves and explain why SGD might catch up after epoch 20 despite slower start.

3. Implement a custom learning rate warmup scheduler that linearly increases LR from 0 to initial LR over the first 10 epochs, then holds constant. Use `LearningRateScheduler` and plot the LR schedule.

**Hard:**
4. Extend the `GradientNormLogger` callback to track gradient norms for each layer separately. Train a 10-layer network and plot gradient norms across layers. Identify where vanishing gradients occur (early or late layers?).

**Bonus:**
5. Build a very deep network (20 layers) on MNIST digits using only Dense layers. Use BatchNorm after every layer and add residual connections (skip connections) using `layers.Add()`. Can you train it without gradients vanishing? Compare to a 20-layer net without residuals.

## Exercise Solutions

<details>
<summary>Click to expand solutions</summary>

**Exercise 2 Solution (Adam vs SGD+Momentum):**
```python
# Adam training
model_adam = build_model()
model_adam.compile(optimizer=Adam(0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_adam = model_adam.fit(x_train, y_train, epochs=30, validation_split=0.2, verbose=0)

# SGD+Momentum training  
model_sgd = build_model()
model_sgd.compile(optimizer=SGD(0.01, momentum=0.9), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
hist_sgd = model_sgd.fit(x_train, y_train, epochs=30, validation_split=0.2, verbose=0)

# Plot comparison
plt.plot(hist_adam.history['val_accuracy'], label='Adam')
plt.plot(hist_sgd.history['val_accuracy'], label='SGD+Momentum')
plt.legend()
```
SGD+momentum often generalizes better because Adam's adaptive learning rates can overshoot sharp minima, while momentum-based methods find wider, more generalizable minima.

**Exercise 3 Solution (Warmup Scheduler):**
```python
def warmup_scheduler(epoch, lr):
    warmup_epochs = 10
    initial_lr = 0.001
    if epoch < warmup_epochs:
        return initial_lr * (epoch + 1) / warmup_epochs
    return initial_lr

callback = LearningRateScheduler(warmup_scheduler)
```

</details>

## Summary – What You Learned Today

**Core Concepts:**
- Backpropagation mechanics: chain rule, forward/backward passes, and why gradients flow the way they do
- Vanishing and exploding gradients: causes, detection, and mitigation strategies

**Optimization Mastery:**
- SGD with momentum accelerates convergence through velocity accumulation
- Adaptive methods (RMSprop, Adam, Nadam) adjust per-parameter learning rates
- AdamW decouples weight decay from gradient updates for better regularization
- Trade-off: Adam converges faster but SGD+momentum often generalizes better

**Regularization Arsenal:**
- L1/L2 regularization controls weight magnitudes and encourages sparsity
- Dropout prevents co-adaptation and forces robust feature learning
- Early stopping halts training at optimal validation performance

**Normalization Techniques:**
- BatchNorm stabilizes training, enables higher learning rates, and reduces initialization sensitivity
- LayerNorm provides batch-independent normalization crucial for sequences and small batches

**Production Training:**
- Callbacks (TensorBoard, ReduceLROnPlateau, ModelCheckpoint) automate training management
- Gradient clipping prevents explosion in deep or recurrent architectures
- Monitoring gradient norms per layer diagnoses training pathologies early

**Practical Skills:**
- Built and trained networks with 10+ layers combining all modern techniques
- Achieved >90% accuracy on Fashion-MNIST with proper regularization
- Implemented custom callbacks for gradient monitoring and learning rate scheduling

---

## Next Notebook Preview

**Notebook 19: Convolutional Neural Networks (CNNs) for Images**

We're moving beyond flattening images! Next time we'll explore:
- **Convolutions**: Sliding filters that detect local patterns (edges, textures, shapes)
- **Pooling**: Spatial downsampling for translation invariance and computation reduction  
- **Feature Hierarchies**: Early layers detect edges, deep layers detect objects
- **Classic Architectures**: LeNet-5, AlexNet fundamentals, and modern design patterns
- **Data Augmentation**: Artificially expand training data with rotations, flips, and crops
- **Transfer Learning**: Leverage pre-trained networks for your own image tasks

Get ready to build computer vision models that actually understand spatial structure! 🖼️